# Experiment Tracking System for Multi-Omics Project
Tracks both successful and failed experiments with detailed logging

In [1]:
import json
import datetime
import os
import hashlib
import pandas as pd
from pathlib import Path
import pickle
from typing import Dict, Any, Optional, List

In [2]:
class ExperimentTracker:
    def __init__(self, project_name="multiomics_tcga", base_dir="./experiments"):
        self.project_name = project_name
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(exist_ok=True)
        
        # Create subdirectories
        self.logs_dir = self.base_dir / "logs"
        self.results_dir = self.base_dir / "results"
        self.failed_dir = self.base_dir / "failed"
        self.models_dir = self.base_dir / "models"
        
        for dir_path in [self.logs_dir, self.results_dir, self.failed_dir, self.models_dir]:
            dir_path.mkdir(exist_ok=True)
        
        self.current_experiment = None
        self.experiment_log = []

    def start_experiment(self, 
                        experiment_name: str,
                        experiment_type: str,
                        description: str,
                        parameters: Dict[str, Any],
                        research_questions: List[str] = None) -> str:
        """Start a new experiment"""
        
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        experiment_id = f"{experiment_type}_{experiment_name}_{timestamp}"
        
        self.current_experiment = {
            "experiment_id": experiment_id,
            "experiment_name": experiment_name,
            "experiment_type": experiment_type,
            "description": description,
            "parameters": parameters,
            "research_questions": research_questions or [],
            "start_time": datetime.datetime.now().isoformat(),
            "status": "running",
            "metrics": {},
            "artifacts": [],
            "logs": [],
            "git_commit": self._get_git_commit(),
            "environment": self._get_environment_info()
        }
        
        print(f"Started experiment: {experiment_id}")
        print(f"Description: {description}")
        print(f"Parameters: {parameters}")
        
        return experiment_id

    def log_metric(self, metric_name: str, value: float, step: int = None):
        """Log a metric value"""
        if self.current_experiment is None:
            raise ValueError("No active experiment. Call start_experiment() first.")
        
        if metric_name not in self.current_experiment["metrics"]:
            self.current_experiment["metrics"][metric_name] = []
        
        metric_entry = {
            "value": value,
            "timestamp": datetime.datetime.now().isoformat(),
            "step": step
        }
        
        self.current_experiment["metrics"][metric_name].append(metric_entry)
        print(f"Logged {metric_name}: {value}" + (f" (step {step})" if step else ""))
    
    def log_message(self, message: str, level: str = "INFO"):
        """Log a message"""
        if self.current_experiment is None:
            raise ValueError("No active experiment. Call start_experiment() first.")
        
        log_entry = {
            "timestamp": datetime.datetime.now().isoformat(),
            "level": level,
            "message": message
        }
        
        self.current_experiment["logs"].append(log_entry)
        print(f"[{level}] {message}")

    def save_artifact(self, artifact_path: str, artifact_type: str, description: str = ""):
        """Save an artifact (model, plot, data file)"""
        if self.current_experiment is None:
            raise ValueError("No active experiment. Call start_experiment() first.")
        
        # Copy artifact to experiment directory
        exp_artifacts_dir = self.results_dir / self.current_experiment["experiment_id"]
        exp_artifacts_dir.mkdir(exist_ok=True)
        
        artifact_filename = Path(artifact_path).name
        destination = exp_artifacts_dir / artifact_filename
        
        # Copy file if it exists
        if os.path.exists(artifact_path):
            import shutil
            shutil.copy2(artifact_path, destination)
        
        artifact_entry = {
            "original_path": artifact_path,
            "saved_path": str(destination),
            "artifact_type": artifact_type,
            "description": description,
            "timestamp": datetime.datetime.now().isoformat()
        }
        
        self.current_experiment["artifacts"].append(artifact_entry)
        print(f"Saved artifact: {artifact_filename} ({artifact_type})")

    def end_experiment(self, status: str = "completed", final_notes: str = ""):
        """End the current experiment"""
        if self.current_experiment is None:
            raise ValueError("No active experiment to end.")
        
        self.current_experiment["end_time"] = datetime.datetime.now().isoformat()
        self.current_experiment["status"] = status
        self.current_experiment["final_notes"] = final_notes
        
        # Calculate duration
        start_time = datetime.datetime.fromisoformat(self.current_experiment["start_time"])
        end_time = datetime.datetime.fromisoformat(self.current_experiment["end_time"])
        self.current_experiment["duration_seconds"] = (end_time - start_time).total_seconds()
        
        # Save experiment log
        if status == "failed":
            log_file = self.failed_dir / f"{self.current_experiment['experiment_id']}.json"
        else:
            log_file = self.logs_dir / f"{self.current_experiment['experiment_id']}.json"
        
        with open(log_file, 'w') as f:
            json.dump(self.current_experiment, f, indent=2)
        
        # Add to experiment history
        self.experiment_log.append(self.current_experiment.copy())
        
        print(f"Experiment ended: {status}")
        print(f"Duration: {self.current_experiment['duration_seconds']:.1f} seconds")
        if final_notes:
            print(f"Notes: {final_notes}")
        
        self.current_experiment = None

    def record_failed_experiment(self, 
                                error_message: str,
                                error_type: str,
                                stack_trace: str = "",
                                attempted_fixes: List[str] = None):
        """Record details of a failed experiment"""
        if self.current_experiment is None:
            raise ValueError("No active experiment to mark as failed.")
        
        failure_details = {
            "error_message": error_message,
            "error_type": error_type,
            "stack_trace": stack_trace,
            "attempted_fixes": attempted_fixes or [],
            "failure_timestamp": datetime.datetime.now().isoformat()
        }
        
        self.current_experiment["failure_details"] = failure_details
        self.log_message(f"Experiment failed: {error_message}", "ERROR")
        
        self.end_experiment(status="failed", 
                          final_notes=f"Failed due to {error_type}: {error_message}")

    def get_experiment_summary(self) -> pd.DataFrame:
        """Get summary of all experiments"""
        if not self.experiment_log:
            return pd.DataFrame()
        
        summary_data = []
        for exp in self.experiment_log:
            summary = {
                "experiment_id": exp["experiment_id"],
                "experiment_name": exp["experiment_name"],
                "experiment_type": exp["experiment_type"],
                "status": exp["status"],
                "start_time": exp["start_time"],
                "duration_seconds": exp.get("duration_seconds", 0),
                "num_metrics": len(exp["metrics"]),
                "num_artifacts": len(exp["artifacts"])
            }
            
            # Add key metrics
            for metric_name, metric_values in exp["metrics"].items():
                if metric_values:
                    summary[f"final_{metric_name}"] = metric_values[-1]["value"]
            
            summary_data.append(summary)
        
        return pd.DataFrame(summary_data)

    def load_experiment(self, experiment_id: str) -> Dict[str, Any]:
        """Load a specific experiment"""
        log_file = self.logs_dir / f"{experiment_id}.json"
        failed_file = self.failed_dir / f"{experiment_id}.json"
        
        if log_file.exists():
            with open(log_file, 'r') as f:
                return json.load(f)
        elif failed_file.exists():
            with open(failed_file, 'r') as f:
                return json.load(f)
        else:
            raise FileNotFoundError(f"Experiment {experiment_id} not found")
    
    def compare_experiments(self, experiment_ids: List[str]) -> pd.DataFrame:
        """Compare multiple experiments"""
        experiments = []
        for exp_id in experiment_ids:
            try:
                exp = self.load_experiment(exp_id)
                experiments.append(exp)
            except FileNotFoundError:
                print(f"Warning: Experiment {exp_id} not found")
        
        if not experiments:
            return pd.DataFrame()
        
        # Create comparison table
        comparison_data = []
        for exp in experiments:
            row = {
                "experiment_id": exp["experiment_id"],
                "experiment_type": exp["experiment_type"],
                "status": exp["status"],
                "duration_seconds": exp.get("duration_seconds", 0)
            }
            
            # Add all metrics
            for metric_name, metric_values in exp["metrics"].items():
                if metric_values:
                    row[metric_name] = metric_values[-1]["value"]
            
            # Add parameters
            for param_name, param_value in exp["parameters"].items():
                row[f"param_{param_name}"] = param_value
            
            comparison_data.append(row)
        
        return pd.DataFrame(comparison_data)

    def _get_git_commit(self) -> Optional[str]:
        """Get current git commit hash"""
        try:
            import subprocess
            result = subprocess.run(['git', 'rev-parse', 'HEAD'], 
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return result.stdout.strip()
        except:
            pass
        return None
    
    def _get_environment_info(self) -> Dict[str, str]:
        """Get environment information"""
        import sys
        import platform
        
        return {
            "python_version": sys.version,
            "platform": platform.platform(),
            "working_directory": os.getcwd()
        }

In [ ]:
# Testing and Demo Code
print("Testing ExperimentTracker system...")

# Initialize experiment tracker
tracker = ExperimentTracker()

print("\n=== Testing Successful Experiment ===")
exp_id = tracker.start_experiment(
    experiment_name="baseline_mrna_vae",
    experiment_type="baseline",
    description="Baseline VAE for mRNA expression data",
    parameters={"latent_dim": 50, "batch_size": 32, "learning_rate": 0.001}
)

tracker.log_message("Starting mRNA VAE training")
tracker.log_metric("training_loss", 0.8, step=1)
tracker.log_metric("training_loss", 0.6, step=2)
tracker.log_metric("validation_accuracy", 0.75)
tracker.save_artifact("model.pkl", "model", "Trained VAE model")
tracker.end_experiment(status="completed")

print("\n=== Testing Failed Experiment ===")
exp_id = tracker.start_experiment(
    experiment_name="failed_fusion_vae",
    experiment_type="multimodal",
    description="Attempting complex fusion VAE",
    parameters={"latent_dim": 100, "fusion_type": "complex"}
)

tracker.log_message("Starting fusion training")
tracker.record_failed_experiment(
    error_message="Out of memory error during training",
    error_type="ValueError",
    attempted_fixes=["Reduce batch size", "Use gradient checkpointing"]
)

print("\n=== Testing Direct Experiment Methods ===")
exp_id = tracker.start_experiment(
    experiment_name="direct_test",
    experiment_type="test",
    description="Testing direct experiment methods",
    parameters={"test_param": "value"}
)

tracker.log_metric("test_metric", 42.0)
tracker.log_message("This is a test message")
tracker.end_experiment(status="completed")

# One more failed experiment for completeness  
exp_id = tracker.start_experiment(
    experiment_name="failed_fusion_vae",
    experiment_type="multimodal",
    description="Attempting complex fusion VAE",
    parameters={"latent_dim": 100, "fusion_type": "complex"}
)

tracker.log_message("Starting fusion training")
tracker.record_failed_experiment(
    error_message="Out of memory error during training",
    error_type="ValueError"
)

print("\n=== Experiment Summary ===")
summary = tracker.get_experiment_summary()
print("Experiment Summary:")
print(summary)

print(f"\nExperiment tracking directories created:")
print(f"Base directory: {tracker.base_dir}")
print(f"Logs: {tracker.logs_dir}")
print(f"Results: {tracker.results_dir}")
print(f"Failed: {tracker.failed_dir}")
print(f"Models: {tracker.models_dir}")

print(f"\n ExperimentTracker is working correctly!")
print(f"Total experiments logged: {len(tracker.experiment_log)}")

if not summary.empty:
    latest_exp = summary.iloc[-1]
    print(f"\nLatest experiment details:")
    print(f"ID: {latest_exp['experiment_id']}")
    print(f"Status: {latest_exp['status']}")
    print(f"Duration: {latest_exp['duration_seconds']:.1f} seconds")